In [0]:
%pip install openai --quiet

In [0]:
dbutils.library.restartPython()

In [0]:

from databricks.sdk import WorkspaceClient
from databricks.sdk.service import jobs
from databricks.sdk.service.workspace import ImportFormat, Language

w = WorkspaceClient()
me = w.current_user.me().user_name
path = f"/Users/{me}/agentic_ai_probe_v2"

w.workspace.upload(
    path=path,
    content=(
        b"# Databricks notebook source\n"
        b"n = spark.table('databricks_ws.agentic_ai_dev.incidents').count()\n"
        b"raise ValueError(f'probe v2: read {n} incident rows, then failed on purpose')\n"
    ),
    format=ImportFormat.SOURCE, language=Language.PYTHON, overwrite=True,
)
job = w.jobs.create(
    name="agentic-ai probe v2 (reads a table, then fails)",
    tasks=[jobs.Task(task_key="read_then_fail", notebook_task=jobs.NotebookTask(notebook_path=path))],
)
run = w.jobs.run_now(job_id=job.job_id)
try:
    run.result()
except Exception as e:
    print("failed as expected:", str(e)[:150])
print("job_id:", job.job_id, "| run_id:", run.run_id)

In [0]:
import os
import sys

here = os.path.dirname(
    dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    .notebookPath().get()
)
probe = "/Workspace" + here
SRC = None
for _ in range(6):
    candidate = os.path.join(probe, "src")
    if os.path.isdir(os.path.join(candidate, "agentic_ai")):
        SRC = candidate
        break
    probe = os.path.dirname(probe)

assert SRC, "could not locate src/agentic_ai above this notebook"
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print("SRC:", SRC)

In [0]:
from agentic_ai.config import Settings
from agentic_ai.telemetry.run_context import build_run_context

s = Settings(catalog="databricks_ws", schema="agentic_ai_dev")
incident = {"job_id": str(job.job_id), "run_id": str(run.run_id), "pipeline_name": "probe v2"}

ctx = build_run_context(s, incident)
print(ctx.to_prompt())
print("\nnotes:", ctx.notes)

In [0]:
display(spark.sql("""
    SELECT event_time, entity_type, entity_id, entity_run_id, source_table_full_name
    FROM system.access.table_lineage
    WHERE source_table_full_name = 'databricks_ws.agentic_ai_dev.incidents'
      AND event_time > current_timestamp() - INTERVAL 7 DAYS
    ORDER BY event_time DESC
    LIMIT 10
"""))

In [0]:
%pip install openai --quiet
from agentic_ai.agents.base import BaseAgent, SubAgent, Tool, ToolContext
from agentic_ai.agents.trace import Trace
from agentic_ai.config import Settings
from agentic_ai.llm.client import LLMClient
from agentic_ai.telemetry.run_context import build_run_context

s = Settings(catalog="databricks_ws", schema="agentic_ai_dev")
incident = {"job_id": "551657987627391", "run_id": "322211937283681", "pipeline_name": "probe v2"}


def row_count(tctx):
    t = tctx.tables[0]
    n = spark.table(t).count()
    return f"{t} is readable and has {n} rows", {"table": t, "rows": n}

class CheckAgent(BaseAgent):
    name = "storage"
    sub_agents = [SubAgent("TABLE_HEALTH", "is the table readable and healthy", (), ("row_count",))]
    tools = {"row_count": Tool("row_count", "count rows", row_count)}

ctx = build_run_context(s, incident)
trace = Trace("INC-FRAMEWORK-CHECK", "probe v2")
d = CheckAgent().run(LLMClient.from_settings(s), ToolContext(s, ctx, spark=spark), trace)

print("trace:", trace.summary())
for step in trace.steps:
    print(f"  {step.step_number}. {step.node_name}: {step.detail[:200]}")
print("\ndiagnosis:", d.diagnosis)
print("requires_human:", d.requires_human, "| complexity:", d.fix_complexity, "| confidence:", d.confidence)
print("flushed:", trace.flush(s, spark))

In [0]:
display(spark.sql(f"""
    SELECT step_number, node_name, sub_agent_selected, left(detail, 150) AS detail, duration_sec
    FROM {s.table('subagent_execution_log')}
    WHERE incident_id = 'INC-FRAMEWORK-CHECK'
    ORDER BY step_number
"""))

In [0]:
%pip install openai

from databricks.sdk import WorkspaceClient
from agentic_ai.agents.base import ToolContext
from agentic_ai.agents.processing import ProcessingAgent
from agentic_ai.agents.router import route
from agentic_ai.agents.trace import Trace
from agentic_ai.config import Settings
from agentic_ai.llm.client import LLMClient
from agentic_ai.telemetry.run_context import build_run_context

w = WorkspaceClient()
probe = next(j for j in w.jobs.list() if (j.settings.name or "").startswith("agentic-ai probe v2"))
latest = next(iter(w.jobs.list_runs(job_id=probe.job_id, limit=1)))
incident = {"job_id": str(probe.job_id), "run_id": str(latest.run_id), "pipeline_name": "probe v2"}
print(incident, "|", latest.state.result_state)

s = Settings(catalog="databricks_ws", schema="agentic_ai_dev")
llm = LLMClient.from_settings(s)
ctx = build_run_context(s, incident)
print("notes:", ctx.notes, "| task errors:", len(ctx.task_errors))

trace = Trace("INC-PROCESSING-CHECK", "probe v2")
r = route(ctx, llm, trace)
print("routed to:", r.agent, "via", r.method)

d = ProcessingAgent().run(llm, ToolContext(s, ctx, spark=spark, workspace_client=w), trace)

print("\ntrace:", trace.summary())
for step in trace.steps:
    print(f"  {step.step_number}. {step.node_name}: {step.detail[:300]}")
print("\ndiagnosis:", d.diagnosis)
print("fix:", d.fix_suggestion)
print("actions:", d.actions)
print("requires_human:", d.requires_human, "| confidence:", d.confidence, "| via:", d.toolkit_used)
print("flushed:", trace.flush(s, spark))